# 03. Prescriptions CSV Extraction (10 Patients)

목표:
- `emr-generator/outputs`, `emr-generator/patient_scenario`를 기준으로 시나리오 타임라인 기간을 계산
- DuckDB `prescriptions`에서 10명(숫자 8자리 patient) 처방 내역 추출
- 출력 컬럼: `patient_id`, `admission_id`, `starttime`(date only), `drug`, `prod_strength`, `route`
- 환자별 CSV 10개 + 요약 CSV 1개 저장


In [26]:
import json
import re
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'emr-generator').exists() and (cand / 'data').exists():
            return cand
    raise FileNotFoundError('project root를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
DB_PATH = PROJECT_ROOT / 'data' / 'mimic_total.duckdb'
EMR_OUTPUTS_DIR = PROJECT_ROOT / 'emr-generator' / 'outputs'
SCENARIO_DIR = PROJECT_ROOT / 'emr-generator' / 'patient_scenario'
TIMELINE_META_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'patient_timelines'  # hadm_id 매핑용
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'prescriptions_by_patient'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DB_PATH: {DB_PATH}')
print(f'EMR_OUTPUTS_DIR: {EMR_OUTPUTS_DIR}')
print(f'SCENARIO_DIR: {SCENARIO_DIR}')
print(f'TIMELINE_META_DIR: {TIMELINE_META_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

if not DB_PATH.exists():
    raise FileNotFoundError(f'DB 파일이 없습니다: {DB_PATH}')


PROJECT_ROOT: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj
DB_PATH: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/data/mimic_total.duckdb
EMR_OUTPUTS_DIR: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/emr-generator/outputs
SCENARIO_DIR: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/emr-generator/patient_scenario
TIMELINE_META_DIR: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/data/outputs/patient_timelines
OUTPUT_DIR: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/data/outputs/prescriptions_by_patient


In [27]:
# emr-generator 시나리오 타임라인 기간(start/end) 계산 + hadm_id 매핑
DATETIME_KEYS = {
    'datetime', 'doc_datetime', 'note_datetime', 'result_datetime', 'collection_datetime',
    'study_datetime', 'event_datetime', 'charttime', 'starttime', 'stoptime'
}

def collect_datetime_values(obj):
    vals = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                vals.extend(collect_datetime_values(v))
            elif k in DATETIME_KEYS and v:
                vals.append(v)
    elif isinstance(obj, list):
        for it in obj:
            vals.extend(collect_datetime_values(it))
    return vals

def get_scenario_title(md_path: Path):
    if not md_path.exists():
        return None
    for line in md_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line.startswith('# '):
            return line[2:].strip()
    return None

# patient_timelines 메타에서 hadm_id 매핑
hadm_map = {}
for path in sorted(TIMELINE_META_DIR.glob('patient_*.json')):
    data = json.loads(path.read_text(encoding='utf-8'))
    pid = str(data.get('subject_id'))
    admissions = data.get('admissions', [])
    if admissions and admissions[0].get('hadm_id') is not None:
        hadm_map[pid] = int(admissions[0]['hadm_id'])

records = []
for patient_dir in sorted(EMR_OUTPUTS_DIR.glob('patient_*')):
    m = re.fullmatch(r'patient_(\d{8})', patient_dir.name)
    if not m:
        continue  # T01~T03 제외

    patient_id = m.group(1)
    admission_id = hadm_map.get(patient_id)

    hd_files = sorted(patient_dir.glob('hd_*.json'))
    dt_values = []
    for f in hd_files:
        payload = json.loads(f.read_text(encoding='utf-8'))
        dt_values.extend(collect_datetime_values(payload.get('documents', [])))

    dt_series = pd.to_datetime(dt_values, errors='coerce')
    dt_series = dt_series[~pd.isna(dt_series)]

    timeline_start = dt_series.min() if len(dt_series) > 0 else pd.NaT
    timeline_end = dt_series.max() if len(dt_series) > 0 else pd.NaT

    scenario_file = SCENARIO_DIR / f'patient_{patient_id}.md'

    records.append({
        'patient_id': patient_id,
        'admission_id': admission_id,
        'timeline_start': timeline_start,
        'timeline_end': timeline_end,
        'timeline_event_count': len(dt_series),
        'hd_file_count': len(hd_files),
        'scenario_file': scenario_file.name if scenario_file.exists() else None,
        'scenario_title': get_scenario_title(scenario_file),
    })

targets_df = pd.DataFrame(records).sort_values('patient_id').reset_index(drop=True)

print(f'대상 환자 수(숫자 8자리): {len(targets_df)}명')
print(f'hadm_id 매핑 성공: {targets_df["admission_id"].notna().sum()}명')
print(f'timeline_start/end 유효: {(targets_df["timeline_start"].notna() & targets_df["timeline_end"].notna()).sum()}명')

display(targets_df[['patient_id','admission_id','timeline_start','timeline_end','timeline_event_count','hd_file_count','scenario_file']])


대상 환자 수(숫자 8자리): 10명
hadm_id 매핑 성공: 10명
timeline_start/end 유효: 10명


,patient_id,admission_id,timeline_start,timeline_end,timeline_event_count,hd_file_count,scenario_file
0,11601773,23610210,2168-03-20 06:00:00,2168-03-23 01:30:00,17,3,patient_11601773.md
1,12249103,28518207,2125-01-28 09:00:00,2125-02-05 01:30:00,53,8,patient_12249103.md
2,12356657,22793142,2155-09-19 10:00:00,2155-09-29 01:30:00,56,8,patient_12356657.md
3,16836931,29338069,2180-10-20 09:30:00,2180-10-26 01:30:00,45,6,patient_16836931.md
4,17650289,21622483,2119-06-10 06:00:00,2119-06-21 01:30:00,63,11,patient_17650289.md
5,18003081,20086040,2181-05-23 08:00:00,2181-05-27 01:30:00,22,4,patient_18003081.md
6,18294629,22456860,2143-09-23 09:30:00,2143-09-30 17:30:00,42,8,patient_18294629.md
7,19096027,20057926,2129-08-09 06:00:00,2129-08-19 01:30:00,54,10,patient_19096027.md
8,19440935,28055582,2126-11-23 06:00:00,2126-12-01 01:30:00,55,8,patient_19440935.md
9,19548143,27966375,2169-09-30 07:00:00,2169-10-17 01:30:00,117,16,patient_19548143.md


In [28]:
# DuckDB 연결 및 prescriptions 컬럼 점검
conn = duckdb.connect(str(DB_PATH), read_only=True)

schema_df = conn.execute('DESCRIBE prescriptions').df()
display(schema_df)

required_columns = ['subject_id', 'hadm_id', 'starttime', 'drug', 'prod_strength', 'route', 'stoptime']
actual_columns = set(schema_df['column_name'].tolist())
missing = [c for c in required_columns if c not in actual_columns]
if missing:
    raise ValueError(f'prescriptions 테이블에 필요한 컬럼이 없습니다: {missing}')

print('필수 컬럼 확인 완료')


,column_name,column_type,null,key,default,extra
0,subject_id,VARCHAR,YES,None,None,None
1,hadm_id,VARCHAR,YES,None,None,None
2,pharmacy_id,VARCHAR,YES,None,None,None
3,poe_id,VARCHAR,YES,None,None,None
4,poe_seq,VARCHAR,YES,None,None,None
5,order_provider_id,VARCHAR,YES,None,None,None
6,starttime,VARCHAR,YES,None,None,None
7,stoptime,VARCHAR,YES,None,None,None
8,drug_type,VARCHAR,YES,None,None,None
9,drug,VARCHAR,YES,None,None,None


필수 컬럼 확인 완료


In [29]:
# 10명 전체 추출 + 환자별 CSV 저장
query = '''
SELECT
    subject_id AS patient_id,
    hadm_id AS admission_id,
    CAST(starttime AS DATE) AS starttime,
    drug,
    prod_strength,
    route
FROM prescriptions
WHERE subject_id = ?
  AND hadm_id = ?
  AND CAST(starttime AS TIMESTAMP) <= CAST(? AS TIMESTAMP)
  AND COALESCE(CAST(stoptime AS TIMESTAMP), CAST(starttime AS TIMESTAMP)) >= CAST(? AS TIMESTAMP)
ORDER BY starttime
'''

valid_targets = targets_df[
    targets_df['admission_id'].notna()
    & targets_df['timeline_start'].notna()
    & targets_df['timeline_end'].notna()
].copy()

print(f'실제 추출 대상: {len(valid_targets)}명')

summary_rows = []
for _, row in valid_targets.iterrows():
    patient_id = int(row['patient_id'])
    admission_id = int(row['admission_id'])

    start_str = pd.to_datetime(row['timeline_start']).strftime('%Y-%m-%d %H:%M:%S')
    end_str = pd.to_datetime(row['timeline_end']).strftime('%Y-%m-%d %H:%M:%S')

    df = conn.execute(query, [patient_id, admission_id, end_str, start_str]).df()

    # 후처리 편의: patient_id/admission_id를 문자열 컬럼으로 통일
    if len(df) > 0:
        df['patient_id'] = df['patient_id'].astype(str)
        df['admission_id'] = df['admission_id'].astype(str)
        df['starttime'] = pd.to_datetime(df['starttime'], errors='coerce').dt.strftime('%Y-%m-%d')

    out_path = OUTPUT_DIR / f'patient_{patient_id}_admission_{admission_id}_prescriptions.csv'
    df.to_csv(out_path, index=False, encoding='utf-8-sig')

    summary_rows.append({
        'patient_id': str(patient_id),
        'admission_id': str(admission_id),
        'timeline_start': start_str,
        'timeline_end': end_str,
        'row_count': len(df),
        'unique_drug_count': df['drug'].nunique() if 'drug' in df.columns else 0,
        'csv_path': str(out_path),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('patient_id').reset_index(drop=True)
summary_csv = OUTPUT_DIR / 'prescriptions_extraction_summary.csv'
summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')

print(f'환자별 CSV 저장 완료: {len(summary_df)}개')
print(f'요약 CSV: {summary_csv}')
display(summary_df)


실제 추출 대상: 10명
환자별 CSV 저장 완료: 10개
요약 CSV: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/data/outputs/prescriptions_by_patient/prescriptions_extraction_summary.csv


,patient_id,admission_id,timeline_start,timeline_end,row_count,unique_drug_count,csv_path
0,11601773,23610210,2168-03-20 06:00:00,2168-03-23 01:30:00,31,28,/Users/iseungmin/Documents/GitHub/oracle-fire4...
1,12249103,28518207,2125-01-28 09:00:00,2125-02-05 01:30:00,146,58,/Users/iseungmin/Documents/GitHub/oracle-fire4...
2,12356657,22793142,2155-09-19 10:00:00,2155-09-29 01:30:00,148,73,/Users/iseungmin/Documents/GitHub/oracle-fire4...
3,16836931,29338069,2180-10-20 09:30:00,2180-10-26 01:30:00,81,45,/Users/iseungmin/Documents/GitHub/oracle-fire4...
4,17650289,21622483,2119-06-10 06:00:00,2119-06-21 01:30:00,45,31,/Users/iseungmin/Documents/GitHub/oracle-fire4...
5,18003081,20086040,2181-05-23 08:00:00,2181-05-27 01:30:00,39,19,/Users/iseungmin/Documents/GitHub/oracle-fire4...
6,18294629,22456860,2143-09-23 09:30:00,2143-09-30 17:30:00,39,24,/Users/iseungmin/Documents/GitHub/oracle-fire4...
7,19096027,20057926,2129-08-09 06:00:00,2129-08-19 01:30:00,152,45,/Users/iseungmin/Documents/GitHub/oracle-fire4...
8,19440935,28055582,2126-11-23 06:00:00,2126-12-01 01:30:00,45,28,/Users/iseungmin/Documents/GitHub/oracle-fire4...
9,19548143,27966375,2169-09-30 07:00:00,2169-10-17 01:30:00,119,55,/Users/iseungmin/Documents/GitHub/oracle-fire4...


In [30]:
# 샘플 확인 (첫 환자 CSV)
if len(summary_df) > 0:
    sample_path = Path(summary_df.iloc[0]['csv_path'])
    print(f'샘플 파일: {sample_path}')
    display(pd.read_csv(sample_path).head(20))
else:
    print('요약 데이터가 없습니다.')


샘플 파일: /Users/iseungmin/Documents/GitHub/oracle-fire4bird/final-prj/data/outputs/prescriptions_by_patient/patient_11601773_admission_23610210_prescriptions.csv


,patient_id,admission_id,starttime,drug,prod_strength,route
0,11601773,23610210,2168-03-09,Nystatin Oral Suspension,"500,000 Unit UDCUP",PO
1,11601773,23610210,2168-03-09,Sodium Chloride 0.9% Flush,Syringe,IV
2,11601773,23610210,2168-03-09,traZODONE,50mg Tablet,PO/NG
3,11601773,23610210,2168-03-09,Aspirin,81mg Tab,PO/NG
4,11601773,23610210,2168-03-09,Pneumococcal Vac Polyvalent,25mcg/0.5mL Vial,IM
5,11601773,23610210,2168-03-09,Thiamine,100mg Tablet,PO/NG
6,11601773,23610210,2168-03-09,Heparin,5000 Units / mL- 1mL Vial,SC
7,11601773,23610210,2168-03-09,Docusate Sodium,100mg Capsule,PO
8,11601773,23610210,2168-03-14,Vial,Send Vial,IV
9,11601773,23610210,2168-03-14,Pantoprazole,40 mg Vial,IV
